### Modelling and Optimization

# 1.import libraries

In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

In [3]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import os

amazon_path = "../data/processed/amazon_cleaned.csv"
mendeley_path = "../data/processed/mendeley_cleaned.csv"

if os.path.exists(amazon_path):
    amazon_df = pd.read_csv(amazon_path)
    mendeley_df = pd.read_csv(mendeley_path)
    print("Files loaded successfully!")
else:
    print(f"File not found at {os.path.abspath(amazon_path)}")

Files loaded successfully!


In [5]:
import os
print("Current Working Directory:", os.getcwd())
print("Files in current directory:", os.listdir())

Current Working Directory: c:\Users\hp\Documents\onmni_sense_ml\notebooks
Files in current directory: ['01_exploratory_data_analysis.ipynb', '02_preprocessing_and_baseline.ipynb', '03_advanced_modeling_and_interpretability.ipynb', '04_final_evaluation_and_conclusions.ipynb']


In [6]:
os.listdir('../data/processed/')

['amazon_cleaned.csv', 'mendeley_cleaned.csv']

# 2.Load Data.

In [7]:
amazon_df = pd.read_csv("../data/processed/amazon_cleaned.csv")
mendeley_df = pd.read_csv("../data/processed/mendeley_cleaned.csv")

# 3. Preprocessing & Sentiment Labeling.
### Here we are Assigning sentiment based on ratings (Amazon) and VADER/Sentiment proxy (Mendeley).
### For Amazon: Ratings >= 4 (Positive), 3 (Neutral), < 3 (Negative)

In [8]:
amazon_df['sentiment'] = pd.cut(amazon_df['ratings'], bins=[0, 2.5, 3.5, 5], labels=['negative', 'neutral', 'positive'])
amazon_df = amazon_df.rename(columns={'name': 'text'})
mendeley_df = mendeley_df.rename(columns={'Review': 'text'})

mendeley_df['sentiment'] = 'positive' 

df = pd.concat([amazon_df[['text', 'sentiment']], mendeley_df[['text', 'sentiment']]], ignore_index=True) # combine and encode
le = LabelEncoder()
df['sentiment_encoded'] = le.fit_transform(df['sentiment'])

X_train, X_test, y_train, y_test = train_test_split(df['text'].fillna(""), df['sentiment_encoded'], test_size=0.2, random_state=42)

# 3. Model Upgrade (Random Forest & XGBoost)
### We use TfidfVectorizer + Classifier in a pipeline
### We compute sample weights to handle bias (the "Fix Bias" task)

In [9]:
weights = compute_sample_weight("balanced", y_train)

# Pipeline Definition
rf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("clf", RandomForestClassifier(class_weight='balanced', random_state=42))
])

xgb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("clf", XGBClassifier(eval_metric='mlogloss'))
])

# 4. Optimization (GridSearchCV)
### Hyperparameter tuning for XGBoost to beat baseline

In [10]:
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [3, 5],
    'clf__learning_rate': [0.01, 0.1]
}
grid_search = GridSearchCV(xgb_pipeline, param_grid, cv=3, scoring='f1_weighted', verbose=1)
grid_search.fit(X_train, y_train, clf__sample_weight=weights)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('tfidf',
                                        TfidfVectorizer(max_features=5000)),
                                       ('clf',
                                        XGBClassifier(base_score=None,
                                                      booster=None,
                                                      callbacks=None,
                                                      colsample_bylevel=None,
                                                      colsample_bynode=None,
                                                      colsample_bytree=None,
                                                      device=None,
                                                      early_stopping_rounds=None,
                                                      enable_categorical=False,
                                                      eval_metric='mlogloss',
                                                      feature_types=None,
                                                      feature_weights=None,
                                                      ga...
                                                      max_cat_threshold=None,
                                                      max_cat_to_onehot=None,
                                                      max_delta_step=None,
                                                      max_depth=None,
                                                      max_leaves=None,
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=None,
                                                      num_parallel_tree=None, ...))]),
             param_grid={'clf__learning_rate': [0.01, 0.1],
                         'clf__max_depth': [3, 5],
                         'clf__n_estimators': [100, 200]},
             scoring='f1_weighted', verbose=1)

# 5. Save Model

In [11]:
joblib.dump(grid_search.best_estimator_, "advanced_model.pkl")
print("Model saved as advanced_model.pkl")


Model saved as advanced_model.pkl
